In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

df = None

In [7]:
path_input = widgets.Text(
    value="../data/sample_data.csv",
    placeholder="Введіть шлях до CSV-файлу",
    description="Шлях:",
    layout=widgets.Layout(width="70%"),
    style={"description_width": "60px"},
)

load_button = widgets.Button(
    description="Завантажити",
    button_style="primary",
    icon="upload",
    layout=widgets.Layout(width="150px"),
)

status_label = widgets.HTML(value="")
preview_output = widgets.Output()

def on_load_clicked(b):
    global df
    path = path_input.value.strip()
    status_label.value = ""
    preview_output.clear_output()

    if not path:
        status_label.value = 'Введіть шлях до файлу'
        return

    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        status_label.value = f'Файл не знайдено: <code>{path}</code>'
        df = None
        return
    except pd.errors.EmptyDataError:
        status_label.value = 'Файл порожній або не містить даних.'
        df = None
        return
    except pd.errors.ParserError as e:
        status_label.value = f'Неможливо прочитати файл (помилка парсингу): {e}'
        df = None
        return
    except Exception as e:
        status_label.value = f'Помилка: {type(e).__name__}: {e}'
        df = None
        return

    if df.empty:
        status_label.value = 'У файлі відсутні дані (порожній DataFrame).'
        df = None
        return

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        status_label.value = 'У файлі немає числових стовпців для аналізу.'
        df = None
        return

    status_label.value = (
        f'Успішно завантажено: '
        f'<b>{len(df)}</b> рядків, <b>{len(df.columns)}</b> стовпців. '
        f'Числові стовпці: {", ".join(numeric_cols)}</p>'
    )
    with preview_output:
        display(df.head())

load_button.on_click(on_load_clicked)

load_ui = widgets.VBox([
    widgets.HBox([path_input, load_button]),
    status_label,
    preview_output,
])
display(load_ui)